In [2]:
import os
import requests

In [3]:
base_url = "https://api.panoramax.xyz"
picture_id ="7ec84f61-fc1e-4114-9d49-07b78b43d9aa"
path = f"api/pictures/{picture_id}"
metadata_url = f"{base_url}/{path}"

output_dir = "streetview_images"
output_filename = f"{picture_id}.jpg"
output_path = os.path.join(output_dir, output_filename)

In [4]:
try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Directory verified: {output_dir}")

    # Get the picture metadata (JSON response)
    print(f"Fetching metadata from: {metadata_url}")
    response = requests.get(metadata_url)
    response.raise_for_status() # Check for HTTP errors
    data = response.json()

    # extract azimuth
    properties = data.get('properties', {})
    
    # 'view:azimuth' is the standard field for camera direction in STAC
    azimuth = properties.get('view:azimuth')

    # Extract the image URL from the response
    # Panoramax/GeoVisio returns a STAC Item. The image links are in the 'assets' dictionary.
    # We prioritize 'hd' (High Definition) but can fall back to 'sd' (Standard Definition).
    assets = data.get('assets', {})
    
    if 'hd' in assets:
        image_url = assets['hd']['href']
        print("Found HD image URL.")
    elif 'sd' in assets:
        image_url = assets['sd']['href']
        print("HD not available, found SD image URL.")
    else:
        raise ValueError("No suitable image asset ('hd' or 'sd') found in the response.")

    print(f"Image URL extracted: {image_url}")

    # Download the actual image content
    print("Downloading image...")
    image_response = requests.get(image_url)
    image_response.raise_for_status()

    # Save to the file system
    with open(output_path, 'wb') as file:
        file.write(image_response.content)

    print(f"Success! Image saved to: {output_path}")

except requests.exceptions.RequestException as e:
    print(f"Network error occurred: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

Directory verified: streetview_images
Fetching metadata from: https://api.panoramax.xyz/api/pictures/7ec84f61-fc1e-4114-9d49-07b78b43d9aa
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/7e/c8/4f/61/fc1e-4114-9d49-07b78b43d9aa.jpg
Success! Image saved to: streetview_images/7ec84f61-fc1e-4114-9d49-07b78b43d9aa.jpg


In [5]:
azimuth

4